## 1. setup

- Imports data manipulation, visualization, preprocessing pipelines, validation tools, and machine learning model libraries.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

print("Setup Complete")

## 2. Data Loading & Missing Value Handling

- Loads the data, filters records missing a target value, transforms prices to a logarithmic scale, and handles physical meaning-based missing values (None or 0).


In [ ]:
# datasets
X_full = pd.read_csv(
    "/kaggle/input/competitions/home-data-for-ml-course/train.csv", index_col="Id"
)
X_test_full = pd.read_csv(
    "/kaggle/input/competitions/home-data-for-ml-course/test.csv", index_col="Id"
)

# Remove rows where the target value is missing
X_full.dropna(axis=0, subset=["SalePrice"], inplace=True)

# Apply log1p transformation to the target variable
y_log = np.log1p(X_full.SalePrice)
X_full.drop(["SalePrice"], axis=1, inplace=True)

# Define columns where missing values represent the absence of a feature (fill with 'None')
none_cols = [
    "PoolQC",
    "MiscFeature",
    "Alley",
    "Fence",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
]
for col in none_cols:
    if col in X_full.columns:
        X_full[col] = X_full[col].fillna("None")
        X_test_full[col] = X_test_full[col].fillna("None")

# Define columns where missing values represent a zero count or measurement (fill with 0)
zero_cols = [
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "BsmtFullBath",
    "BsmtHalfBath",
]
for col in zero_cols:
    if col in X_full.columns:
        X_full[col] = X_full[col].fillna(0)
        X_test_full[col] = X_test_full[col].fillna(0)

# Display dataset dimensions
print(f"Train set shape: {X_full.shape}")
print(f"Test set shape: {X_test_full.shape}")

""" >>>
Train set shape: (1460, 79)
Test set shape: (1459, 79)
"""

## 3. Feature Engineering

- Converts internal class keys to strings and engineers performance-driving attributes including total living space, bathroom counts, building age, and outdoor area.


In [ ]:
def add_custom_features(df):
    df_out = df.copy()

    if "MSSubClass" in df_out.columns:
        df_out["MSSubClass"] = df_out["MSSubClass"].astype(str)

    # -- Re-apply constant missing value imputation logic to guarantee consistency --
    none_cols = [
        "PoolQC",
        "MiscFeature",
        "Alley",
        "Fence",
        "FireplaceQu",
        "GarageType",
        "GarageFinish",
        "GarageQual",
        "GarageCond",
        "BsmtQual",
        "BsmtCond",
        "BsmtExposure",
        "BsmtFinType1",
        "BsmtFinType2",
    ]
    for col in none_cols:
        if col in df_out.columns:
            df_out[col] = df_out[col].fillna("None")

    zero_cols = [
        "GarageArea",
        "GarageCars",
        "BsmtFinSF1",
        "BsmtFinSF2",
        "BsmtUnfSF",
        "TotalBsmtSF",
        "BsmtFullBath",
        "BsmtHalfBath",
    ]
    for col in zero_cols:
        if col in df_out.columns:
            df_out[col] = df_out[col].fillna(0)

    # Calculate Total Living Space Area
    df_out["TotalSF"] = df_out["TotalBsmtSF"] + df_out["1stFlrSF"] + df_out["2ndFlrSF"]
    # Calculate Total Bathroom Counts (combining full and half baths)
    df_out["TotalBaths"] = (
        df_out["FullBath"]
        + (0.5 * df_out["HalfBath"])
        + df_out["BsmtFullBath"]
        + (0.5 * df_out["BsmtHalfBath"])
    )
    # Compute House Age and Remodeling Age based on year sold
    df_out["HouseAge"] = df_out["YrSold"] - df_out["YearBuilt"]
    df_out["RemodAge"] = df_out["YrSold"] - df_out["YearRemodAdd"]
    # Sum up all outdoor deck and porch square footage
    df_out["TotalOutsideSF"] = (
        df_out["WoodDeckSF"]
        + df_out["OpenPorchSF"]
        + df_out["EnclosedPorch"]
        + df_out["3SsnPorch"]
        + df_out["ScreenPorch"]
    )
    return df_out


# Run custom feature engineering on training and test datasets
X_full_fe = add_custom_features(X_full)
X_test_full_fe = add_custom_features(X_test_full)

## 4. Feature Selection & Grouping

- Groups columns based on data types and value order dependencies. Explicitly maps ordered performance rating chains from "Poor" to "Excellent".


In [ ]:
# Identify features with a strict rank or quality progression
ordinal_cols = [
    "ExterQual",
    "ExterCond",
    "BsmtQual",
    "BsmtCond",
    "HeatingQC",
    "KitchenQual",
    "FireplaceQu",
    "GarageQual",
    "GarageCond",
]

# Define quality levels (from Po=Poor to Ex=Excellent)
qual_order = ["None", "Po", "Fa", "TA", "Gd", "Ex"]
ordinal_categories = [qual_order for _ in ordinal_cols]

# -- Extract nominal categorical columns (excluding columns handled by OrdinalEncoder) --
categorical_cols = [
    cname
    for cname in X_full_fe.columns
    if X_full_fe[cname].dtype == "object" and cname not in ordinal_cols
]
numerical_cols = [
    cname
    for cname in X_full_fe.columns
    if X_full_fe[cname].dtype in ["int64", "float64"]
]

# Consolidate target features and filter datasets
my_cols = numerical_cols + ordinal_cols + categorical_cols
X = X_full_fe[my_cols].copy()
X_test = X_test_full_fe[my_cols].copy()

print(f"Selected numerical features: {len(numerical_cols)}")
print(f"Selected categorical features: {len(categorical_cols)}")

""" >>>
Selected numerical features: 40
Selected categorical features: 35
"""

## 5. Preprocessing Pipelines

- Integrates isolated pre-processing strategies (imputation, scaling, one-hot encoding, ordinal transformation) together via an unified ColumnTransformer.


In [ ]:
# Numerical Pipeline: Impute missing entries with median value + apply standard scaling
num_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Ordinal Pipeline: Impute with most frequent value + encode via designated rank list
ord_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "ordinal",
            OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

# Unordered Categorical Pipeline: Impute with most frequent + apply One-Hot Encoding
cat_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

# Combine pipelines into a unified data preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, numerical_cols),
        ("ord", ord_transformer, ordinal_cols),
        ("cat", cat_transformer, categorical_cols),
    ]
)

print(f"X Train shape: {X.shape}")
print(f"X Test shape: {X_test.shape}")
assert list(X.columns) == list(X_test.columns), (
    "Warning: Train and test features do not align!"
)

""" >>>
X Train shape: (1460, 84)
X Test shape: (1459, 84)
"""

## 6. Train-Test Split

- Parts training instances and log prices into standard training (80%) and localized validation (20%) matrices.


In [ ]:
# Partition into 80% training data and 20% validation split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y_log, train_size=0.8, test_size=0.2, random_state=0
)

## 7. Model Training & Initial Validation

- Fits a basic Random Forest and an initial XGBoost pipeline using the training split, calculating true-price scale MAE on the validation split.


In [ ]:
# -- Construct and fit a Random Forest Pipeline --
rf_model = RandomForestRegressor(n_estimators=100, random_state=0)
rf_pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", rf_model)])
rf_pipeline.fit(X_train, y_train)

# Predict validation values and convert back to original price scale to compute MAE
rf_preds = np.expm1(rf_pipeline.predict(X_valid))
rf_mae = mean_absolute_error(np.expm1(y_valid), rf_preds)
print(f"Random Forest Validation MAE: {rf_mae:,.2f}")

# -- Construct and fit an initial XGBoost Pipeline --
xgb_model = XGBRegressor(n_estimators=1000, learning_rate=0.05, random_state=0)
xgb_pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", xgb_model)])
xgb_pipeline.fit(X_train, y_train)

xgb_preds = np.expm1(xgb_pipeline.predict(X_valid))
xgb_mae = mean_absolute_error(np.expm1(y_valid), xgb_preds)
print(f"XGBoost Validation MAE: {xgb_mae:,.2f}")

""" >>>
Random Forest Validation MAE: 17,350.54
XGBoost Validation MAE: 17,328.13
"""

## 8. Hyperparameter Optimization & Cross-Validation

- Searches through a list of candidate estimators using 5-fold cross-validation, selects the parameter with the lowest error, and plots the results.


In [ ]:
# Function calculating mean MAE across 5-fold cross-validation given tree count
def get_cv_score(n_estimators):
    model = XGBRegressor(
        n_estimators=n_estimators, learning_rate=0.05, max_depth=6, random_state=0
    )
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])

    # Calculate negative MAE metrics using target y_log arrays
    scores = -1 * cross_val_score(
        pipeline, X, y_log, cv=5, scoring="neg_mean_absolute_error"
    )
    return scores.mean()


candidate_trees = [100, 250, 500, 750, 1000]
results = {n: get_cv_score(n) for n in candidate_trees}
for n, score in results.items():
    print(f"n_estimators={n} -> CV MAE: {score:,.2f}")

# Extract the parameters that minimize the cross-validation error
best_n_estimators = min(results, key=results.get)
print(
    f"\n Best tree count: {best_n_estimators}, Lowest MAE: {results[best_n_estimators]:,.2f}"
)

""" >>>
n_estimators=100 -> CV MAE: 0.09
n_estimators=250 -> CV MAE: 0.09
n_estimators=500 -> CV MAE: 0.09
n_estimators=750 -> CV MAE: 0.09
n_estimators=1000 -> CV MAE: 0.09

Best tree count: 750, Lowest MAE: 0.09
"""

In [ ]:
# Plot hyperparameter tuning curve trends
plt.figure(figsize=(8, 4))
plt.plot(list(results.keys()), list(results.values()), marker="o")
plt.xlabel("n_estimators")
plt.ylabel("MAE")
plt.title("XGBoost Parameter Tuning")
plt.grid(True)
plt.show()

## 9. Final Model Retraining

- Updates the production XGBoost configuration with the optimized hyperparameter, then fits it over the full dataset.


In [ ]:
# Define final model using best optimized hyperparameter settings
final_model = XGBRegressor(
    n_estimators=best_n_estimators, learning_rate=0.05, max_depth=6, random_state=0
)
final_pipeline = Pipeline(
    steps=[("preprocessor", preprocessor), ("model", final_model)]
)

# Retrain the final pipeline model using all available data
final_pipeline.fit(X, y_log)
print("Final model full training complete!")

## 10. Test Inferences & CSV Submission Export

- Runs test predictions, handles exponential reversal, validates data consistency, and exports outputs to a target submission.csv file.


In [ ]:
# Predict on un-labeled test dataset and convert scale back via expm1
preds_test_log = final_pipeline.predict(X_test)
preds_test = np.expm1(preds_test_log)

print("Are there any nulls in test predictions?:", np.isnan(preds_test).any())
print(
    "Descriptive statistics for predicted prices:\\n", pd.Series(preds_test).describe()
)

""" >>>
Are there any nulls in test predictions?: False
Descriptive statistics for predicted prices:
 count      1459.000000
mean     177761.765625
std       75746.070312
min       46898.042969
25%      128724.242188
50%      158217.171875
75%      207776.625000
max      578100.500000
dtype: float64
"""

In [ ]:
# Structure target DataFrame submission arrays
output = pd.DataFrame({"Id": X_test.index, "SalePrice": preds_test})

output.to_csv("submission.csv", index=False)
print("submission.csv successfully exported! Format as follows:")
print(output.head())

""" >>>
submission.csv successfully exported! Format as follows:
     Id      SalePrice
0  1461  123401.250000
1  1462  170288.093750
2  1463  176094.890625
3  1464  188095.703125
4  1465  187228.781250
"""